# Solution — Fraud Detection Pipeline: Consumer

Reads from `transactions`, enriches each message, and routes output to:
- **`alerts` topic** — HIGH and CRITICAL transactions (for a downstream notification service)
- **`account-stats` topic** — a session snapshot every 10 messages (for a downstream dashboard)
- **`alerts.log` file** — HIGH and CRITICAL with masked account IDs

> Start this before running the producer so messages are consumed as they arrive.

In [ ]:
from kafka import KafkaConsumer, KafkaProducer
from collections import deque, Counter
from datetime import datetime
import json

# ----- Topic & broker configuration -----
TOPIC_IN    = 'transactions'
TOPIC_ALERT = 'alerts'
TOPIC_STATS = 'account-stats'
BROKERS     = ['course-kafka:9092']
ALERTS_FILE = '/home/developer/kafka/sinkFiles/alerts.log'

# ----- Business logic constants -----
EXCHANGE_RATES   = {'ILS': 1.0, 'USD': 3.7, 'EUR': 4.0}
RISK_EMOJI       = {'LOW': '🔵', 'MEDIUM': '🟡', 'HIGH': '🔴', 'CRITICAL': '⚫'}
VELOCITY_WINDOW  = 20      # sliding window size (number of recent messages)
VELOCITY_LIMIT   = 5       # flag if same account appears this many times in the window
SUMMARY_EVERY    = 10      # publish a stats snapshot every N messages

In [ ]:
# ── Transformation helpers ────────────────────────────────────────────────────

def classify_risk(amount_ils: float) -> str:
    """Map a normalized ILS amount to one of four risk levels."""
    if amount_ils < 500:    return 'LOW'
    elif amount_ils < 2000: return 'MEDIUM'
    elif amount_ils < 5000: return 'HIGH'
    else:                   return 'CRITICAL'


def mask_account(account_id: str) -> str:
    """
    Mask the numeric suffix of an account ID for privacy in logs and alert events.
    'ACC_3' → 'ACC_***'
    split('_') gives ['ACC', '3'], so [0] is always the prefix regardless of suffix length.
    """
    return account_id.split('_')[0] + '_***'

In [ ]:
# ── Kafka clients ─────────────────────────────────────────────────────────────

# Consumer: reads from the raw transactions topic.
# group_id='fraud-monitor' ensures offset is committed so the consumer
# resumes from where it left off after a restart.
consumer = KafkaConsumer(
    TOPIC_IN,
    group_id                = 'fraud-monitor',
    bootstrap_servers       = BROKERS,
    auto_commit_interval_ms = 1000
)

# alerts_producer: publishes enriched HIGH/CRITICAL events to the 'alerts' topic.
# A separate downstream service (e.g. an SMS notifier) would subscribe to this topic.
alerts_producer = KafkaProducer(
    bootstrap_servers = BROKERS,
    acks              = 1,
    retries           = 3
)

# stats_producer: publishes session snapshots to the 'account-stats' topic.
# A dashboard or BI tool would consume this topic to display live account metrics.
stats_producer = KafkaProducer(
    bootstrap_servers = BROKERS,
    acks              = 1,
    retries           = 3
)

In [ ]:
# ── Session state (in-memory, reset on restart) ───────────────────────────────

account_counts   = {}    # { 'ACC_1': 5, ... }
account_totals   = {}    # { 'ACC_1': 12500.00, ... }
account_type_log = {}    # { 'ACC_1': Counter({'PURCHASE': 3, ...}), ... }
risk_counts      = {'LOW': 0, 'MEDIUM': 0, 'HIGH': 0, 'CRITICAL': 0}
velocity_alerts  = 0
total_messages   = 0

# deque(maxlen=20): fixed-size sliding window over the last 20 account_ids.
# When a 21st item is appended, the oldest is automatically discarded —
# no manual slicing or pop() needed. This is the standard Python pattern
# for sliding-window operations.
recent_accounts = deque(maxlen=VELOCITY_WINDOW)


def build_stats_payload() -> dict:
    """Build the JSON-serializable snapshot dict for the account-stats topic."""
    accounts_data = {}
    for acc in sorted(account_counts):
        n     = account_counts[acc]
        total = account_totals[acc]
        # max() with key=counter.get returns the key with the highest count.
        top_t = max(account_type_log[acc], key=account_type_log[acc].get)
        accounts_data[acc] = {
            'transactions': n,
            'total_ils':    round(total, 2),
            'avg_ils':      round(total / n, 2),
            'top_type':     top_t
        }
    return {
        'snapshot_at':     total_messages,
        'timestamp':       datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'accounts':        accounts_data,
        'risk_breakdown':  risk_counts.copy(),
        'velocity_alerts': velocity_alerts
    }

In [ ]:
# ── Main consumer loop ────────────────────────────────────────────────────────

with open(ALERTS_FILE, 'w') as alert_file:

    for message in consumer:

        # ── 1. Deserialize ───────────────────────────────────────────────────
        # The producer sent json.dumps(...).encode('utf-8'), so we reverse:
        # bytes → str → dict
        data = json.loads(message.value.decode('utf-8'))

        acc     = data['account_id']
        amount  = data['amount']
        currency= data['currency']
        tx_type = data['transaction_type']
        city    = data['city']
        ts      = data['timestamp']
        merchant= data.get('merchant')   # None for non-PURCHASE types

        # ── 2. Normalize currency to ILS ─────────────────────────────────────
        amount_ils = round(amount * EXCHANGE_RATES[currency], 2)
        original   = f"{amount:.2f} {currency}"   # e.g. "142.50 USD" (always 2 decimals)

        # Enrich the original dict with the new fields (spec 2.1)
        data['amount_ils'] = amount_ils
        data['original']   = original

        # ── 3. Classify risk ─────────────────────────────────────────────────
        risk = classify_risk(amount_ils)

        total_messages += 1

        # ── 4. Update in-memory stats ────────────────────────────────────────
        account_counts[acc] = account_counts.get(acc, 0) + 1
        account_totals[acc] = account_totals.get(acc, 0.0) + amount_ils
        if acc not in account_type_log:
            account_type_log[acc] = Counter()
        account_type_log[acc][tx_type] += 1
        risk_counts[risk] += 1

        # ── 5. Velocity fraud detection ──────────────────────────────────────
        # Append current account to the sliding window, then count occurrences.
        # deque handles the window automatically — oldest item is dropped when
        # the deque exceeds maxlen.
        recent_accounts.append(acc)
        count_in_window = recent_accounts.count(acc)
        if count_in_window >= VELOCITY_LIMIT:
            velocity_alerts += 1
            print(f"⚠️  VELOCITY ALERT | {acc} appeared {count_in_window} times in the last {VELOCITY_WINDOW} transactions")

        # ── 6. Console output ────────────────────────────────────────────────
        # Note: account_id is NOT masked here — masking only applies to stored/published data.
        merchant_str = f"@ {merchant}" if merchant else ''
        print(
            f"{RISK_EMOJI[risk]} {risk:<8} | {acc} | "
            f"{original} → {amount_ils:.2f} ILS | "
            f"{tx_type:<12} {merchant_str:<15}| {city}"
        )

        # ── 7. Route HIGH + CRITICAL events ─────────────────────────────────
        if risk in ('HIGH', 'CRITICAL'):

            # Build the alert payload with the masked account ID.
            alert_payload = {
                'account_id':       mask_account(acc),   # ← masked for privacy
                'amount_ils':       amount_ils,
                'original':         original,
                'risk':             risk,
                'transaction_type': tx_type,
                'city':             city,
                'timestamp':        ts
            }

            # Publish to the 'alerts' topic.
            # Using risk as the key lets downstream consumers filter by risk level
            # without deserializing the full payload — a common Kafka optimization.
            alerts_producer.send(
                topic = TOPIC_ALERT,
                key   = risk.encode('utf-8'),
                value = json.dumps(alert_payload).encode('utf-8')
            )
            alerts_producer.flush()

            # Also write to the local alert log file (masked).
            log_line = (
                f"[{ts}] {risk} | {mask_account(acc)} | "
                f"{original} → {amount_ils:.2f} ILS | {tx_type} | {city}"
            )
            alert_file.write(log_line + '\n')
            # flush() forces the OS write buffer to disk immediately.
            # Without this, alerts may be lost if the process crashes.
            alert_file.flush()

        # ── 8. Publish stats snapshot every N messages ───────────────────────
        if total_messages % SUMMARY_EVERY == 0:
            stats_payload = build_stats_payload()

            # Using a fixed key 'summary' means all snapshots go to the same
            # partition, preserving their chronological order for consumers.
            stats_producer.send(
                topic = TOPIC_STATS,
                key   = b'summary',
                value = json.dumps(stats_payload).encode('utf-8')
            )
            stats_producer.flush()
            print(f"\n📊 Snapshot #{total_messages // SUMMARY_EVERY} published to '{TOPIC_STATS}'\n")